# 01 — Exploratory Data Analysis

**Dataset:** Ames Housing (OpenML id 42165), fetched by `scripts/download_data.py` with a schema guard.

**Scope note:** all EDA below runs on the **training split only** (1168 rows). The 292-row holdout stays unseen — even exploratory looks would leak information into decisions about transforms, outliers, and encodings. The holdout is touched exactly once, at final evaluation.

This notebook contains narrative only; all computation lives in `house_price.visualization`.

In [ ]:
%matplotlib inline
from house_price.config import load_config
from house_price.data import load_split
from house_price import visualization as viz

cfg = load_config()
cfg.paths.ensure()
train_df, _ = load_split(cfg)  # holdout deliberately discarded here
FIG = cfg.paths.figures
train_df.shape

## 1. Target distribution

`SalePrice` is strongly right-skewed — a handful of expensive houses stretch the tail. Modelling the raw target makes squared-error losses obsess over those few houses. `log1p` compresses the tail toward symmetry, making errors proportional (being \$20k off on a \$100k house is worse than on a \$700k house) and satisfying the roughly-Gaussian-residual assumption of the linear models. This confirms the `log1p` decision in config.

In [ ]:
viz.plot_target_distribution(train_df[cfg.dataset.target], save_path=FIG / "target_distribution.png");

## 2. Missingness — informative vs true gaps

Two very different phenomena hide behind NaN in this dataset:

- **Informative missingness** (blue): per the Ames data dictionary, NaN in `PoolQC`, `Alley`, `Fence`, the garage/basement quality columns etc. means *the feature does not exist* (no pool, no alley). Imputing a "typical" value here would be actively wrong — a house without a pool is not a house with an average pool. Preprocessing will map these to an explicit `"None"` category (or 0 for `GarageYrBlt`).
- **True gaps** (red): `LotFrontage` NaNs are genuinely unrecorded street-frontage measurements. These get median imputation inside the pipeline (grouped by neighborhood, since frontage is strongly neighborhood-dependent).

In [ ]:
missing = viz.missingness_table(train_df)
viz.plot_missingness(missing, save_path=FIG / "missingness.png")
missing

## 3. Feature skewness

Several area-type features (lot area, porch areas, basement finish) are heavily right-skewed, mostly because zero is a legitimate and common value ("no porch"). These are transform candidates for the linear models; tree models are indifferent to monotone transforms, which is one reason the encoding/transform comparison in M4 is run per model family.

In [ ]:
skew = viz.numeric_skewness(train_df.drop(columns=[cfg.dataset.target, "Id"]))
viz.plot_skewed_features(train_df, skew.head(9).index.tolist(), save_path=FIG / "skewed_features.png")
skew.head(15)

## 4. Correlations with the target

`OverallQual` and `GrLivArea` dominate, followed by garage and basement size and the year columns. Note the correlated clusters (`GarageCars`/`GarageArea`, `TotalBsmtSF`/`1stFlrSF`) — multicollinearity that regularised linear models handle and that motivates aggregate engineered features over keeping every raw variant.

In [ ]:
viz.plot_correlation_heatmap(train_df.drop(columns=["Id"]), cfg.dataset.target, top_n=15, save_path=FIG / "correlation_heatmap.png")
viz.target_correlations(train_df.drop(columns=["Id"]), cfg.dataset.target).head(15)

## 5. Outlier study

The Ames dataset is known for a few very large houses sold unusually cheaply (documented by the dataset author as partial sales). They sit far off the `GrLivArea`–`SalePrice` trend and can dominate squared-error fits. Decision deferred to M3: rather than silently dropping rows, we will evaluate models with and without a documented outlier filter and keep whichever is justified by CV evidence.

In [ ]:
viz.plot_outlier_scatter(train_df, "GrLivArea", cfg.dataset.target, save_path=FIG / "outliers_grlivarea.png")
train_df.loc[(train_df["GrLivArea"] > 4000) & (train_df[cfg.dataset.target] < 300000), ["Id", "GrLivArea", "OverallQual", "SalePrice"]]

## 6. Neighborhood structure

Median prices vary by a factor of ~3 across neighborhoods with wildly different spreads — strong predictive signal, but 25 categories is too many for OHE to use efficiently in small data. This is exactly why `Neighborhood` is the designated candidate for CV-safe target encoding in the M4 comparison.

In [ ]:
viz.plot_neighborhood_prices(train_df, cfg.dataset.target, save_path=FIG / "neighborhood_prices.png");

## Findings → decisions carried into M3/M4

1. **Target:** right-skew confirmed → keep `log1p` (config).
2. **Missingness:** informative-NA columns → explicit `"None"`/0; `LotFrontage` → neighborhood-grouped median imputation, inside the pipeline.
3. **Skew:** log-transform skewed numerics for linear models only (tree models unaffected).
4. **Collinear clusters** (garage, basement/floor) → prefer engineered aggregates (`TotalSF`, `TotalBaths`).
5. **Outliers:** two partial-sale rows; decide by CV comparison in M3/M4, documented either way.
6. **Neighborhood:** high-cardinality, high-signal → target-encoding candidate (M4).
7. **`Id`** is an index, not a feature → dropped in preprocessing.